# Practice Exercise 3 (Additional): K-Means Image Segmentation

> **Problem:** Segment a given natural image into K color clusters using the K-Means algorithm from scratch.
> **Dataset:** Natural landscape image (`data/raw/nature_landscape.jpg`).


## 1. Load the Image
Read image using PIL or OpenCV and convert to RGB.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Add root to sys.path
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from src.training.kmeans import KMeansScratch

image_path = ROOT / "data" / "raw" / "nature_landscape.jpg"
img = Image.open(image_path).convert("RGB")
print("Image Loaded. Original Size:", img.size)
plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.title("Original Image")
plt.axis("off")
plt.show()

## 2. Preprocess the Image
Resize to smaller size for faster processing and flatten to 2D array of pixels.

In [ ]:
img_resized = img.copy()
img_resized.thumbnail((150, 150), Image.Resampling.LANCZOS)
img_array = np.array(img_resized, dtype=np.float64) / 255.0
h, w, c = img_array.shape
pixel_features = img_array.reshape(-1, c)
print(f"Resized to: {w}x{h}")
print(f"Flattened 2D Pixel Features Shape: {pixel_features.shape}")

## 3 & 4. Apply K-Means Clustering & Segment the Image
Apply `KMeansScratch` to cluster pixels into K centroid colors, replace pixels, and reshape back.

In [ ]:
k_values = [2, 3, 5, 8]
segmented_images = []
inertias = []

for k in k_values:
    kmeans = KMeansScratch(n_clusters=k, init="k-means++", max_iter=100, random_state=42)
    kmeans.fit(pixel_features)
    centroids = kmeans.cluster_centers_
    labels = kmeans.labels_
    
    # Replace each pixel with centroid color
    seg_pixels = centroids[labels]
    seg_img = (seg_pixels.reshape(h, w, c) * 255.0).clip(0, 255).astype(np.uint8)
    segmented_images.append(seg_img)
    inertias.append(kmeans.inertia_)
    print(f"Done K={k} | WCSS (Inertia): {kmeans.inertia_:.2f} | Centroids shape: {centroids.shape}")

## 5. Visualize Results Side-by-Side
Display original and segmented images side-by-side for comparison.

In [ ]:
fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(18, 4))
axes[0].imshow(np.array(img_resized))
axes[0].set_title(f"Original\n{w}x{h}", fontsize=11, fontweight="bold")
axes[0].axis("off")

for idx, (k, seg_img, inr) in enumerate(zip(k_values, segmented_images, inertias), start=1):
    axes[idx].imshow(seg_img)
    axes[idx].set_title(f"K={k} Clusters\nInertia: {inr:.1f}", fontsize=11, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("K-Means Image Segmentation Side-by-Side Comparison", fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()